In [ ]:
import tqdm
import matplotlib.pyplot as plt
import numpy as np
from numpy import pi as π
import firedrake
from firedrake import as_vector, inner, ds
import icepack, icepack.plot

In [ ]:
import pygmsh

R = 200e3
δx = 5e3

In [ ]:
geometry = pygmsh.built_in.Geometry()

x1 = geometry.add_point([-R, 0, 0], lcar=δx)
x2 = geometry.add_point([+R, 0, 0], lcar=δx)

center1 = geometry.add_point([0, 0, 0,], lcar=δx)
center2 = geometry.add_point([0, -4 * R, 0], lcar=δx)

arcs = [geometry.add_circle_arc(x1, center1, x2),
        geometry.add_circle_arc(x2, center2, x1)]

In [ ]:
line_loop = geometry.add_line_loop(arcs)
plane_surface = geometry.add_plane_surface(line_loop)

physical_lines = [geometry.add_physical(arc) for arc in arcs]
physical_surface = geometry.add_physical(plane_surface)

In [ ]:
with open('ice-shelf.geo', 'w') as geo_file:
    geo_file.write(geometry.get_code())

In [ ]:
!gmsh -2 -format msh2 -o ice-shelf.msh ice-shelf.geo

In [ ]:
mesh = firedrake.Mesh('ice-shelf.msh')

In [ ]:
inlet_angles = π * np.array([-3/4, -1/2, -1/3, -1/6])
inlet_widths = π * np.array([1/8, 1/12, 1/24, 1/12])

In [ ]:
def make_symbolic_input_data(x):
    u_in = 300
    h_in = 350
    hb = 100
    dh, du = 400, 250

    hs, us = [], []
    for θ, ϕ in zip(inlet_angles, inlet_widths):
        x0 = R * as_vector((np.cos(θ), np.sin(θ)))
        v = -as_vector((np.cos(θ), np.sin(θ)))
        L = inner(x - x0, v)
        W = x - x0 - L * v
        Rn = 2 * ϕ / π * R
        q = firedrake.max_value(1 - (W / Rn)**2, 0)
        hs.append(hb + q * ((h_in - hb) - dh * L /R))
        us.append(firedrake.exp(-4 * (W/R)**2) * (u_in + du * L / R) * v)
        
    h_expr = firedrake.Constant(hb)
    for h in hs:
        h_expr = firedrake.max_value(h, h_expr)
    
    u_expr = sum(us)
    
    return h_expr, u_expr

In [ ]:
x = firedrake.SpatialCoordinate(mesh)
h_expr, u_expr = make_symbolic_input_data(x)

In [ ]:
Q1 = firedrake.FunctionSpace(mesh, family='CG', degree=1)
V1 = firedrake.VectorFunctionSpace(mesh, family='CG', degree=1)

Q2 = firedrake.FunctionSpace(mesh, family='CG', degree=2)
V2 = firedrake.VectorFunctionSpace(mesh, family='CG', degree=2)

fields1 = {
    'thickness': firedrake.interpolate(h_expr, Q1),
    'velocity': firedrake.interpolate(u_expr, V1)
}

fields2 = {
    'thickness': firedrake.interpolate(h_expr, Q2),
    'velocity': firedrake.interpolate(u_expr, V2)
}

In [ ]:
model = icepack.models.IceShelf()

In [ ]:
solver1 = icepack.solvers.FlowSolver(model, dirichlet_ids=[1])
solver2 = icepack.solvers.FlowSolver(model, dirichlet_ids=[1])

In [ ]:
T = firedrake.Constant(255.15)
A = icepack.rate_factor(T)

In [ ]:
h1 = fields1['thickness'].copy(deepcopy=True)
fields1['velocity'] = solver1.diagnostic_solve(**fields1, fluidity=A)

h2 = fields2['thickness'].copy(deepcopy=True)
fields2['velocity'] = solver2.diagnostic_solve(**fields2, fluidity=A)

In [ ]:
final_time = 400.
num_timesteps = 200
dt = final_time / num_timesteps
a = firedrake.Constant(0.0)

for step in tqdm.trange(num_timesteps):
    fields1['thickness'] = solver1.prognostic_solve(
        dt, **fields1, accumulation=a, thickness_inflow=h1
    )
    
    fields1['velocity'] = solver1.diagnostic_solve(
        **fields1, fluidity=A
    )
    
    fields2['thickness'] = solver2.prognostic_solve(
        dt, **fields2, accumulation=a, thickness_inflow=h2
    )
    
    fields2['velocity'] = solver2.diagnostic_solve(
        **fields2, fluidity=A
    )

### Adapting the mesh

In [ ]:
δh = firedrake.interpolate(
    fields2['thickness'] - fields1['thickness'], Q2
)

In [ ]:
from firedrake import dx
volume = firedrake.assemble(fields2['thickness']**2 * dx)
firedrake.assemble(δh**2 * dx) / volume

In [ ]:
fig, axes = icepack.plot.subplots()
triangles = icepack.plot.tripcolor(
    δh, vmin=-10, vmax=+10, shading='gouraud', cmap='RdBu_r', axes=axes
)
fig.colorbar(triangles)

In [ ]:
from firedrake import inner, grad, dx
m = firedrake.Function(Q2)
α = firedrake.Constant(12e3)
J = ((m - abs(δh))**2 + α**2 * inner(grad(m), grad(m))) * dx
F = firedrake.derivative(J, m)
parameters = {
    'solver_parameters': {
        'ksp_type': 'preonly',
        'pc_type': 'lu'
    }
}
firedrake.solve(F == 0, m, **parameters)

In [ ]:
fig, axes = icepack.plot.subplots()
triangles = firedrake.tripcolor(m, shading='gouraud', axes=axes)
fig.colorbar(triangles)

In [ ]:
M = m.dat.data_ro[:]
δm = M.max() - M.min()
s = firedrake.Constant(16.)
m.assign(1 + (s - 1) * m / δm);

In [ ]:
monge_ampere_solver = icepack.adaptivity.PseudoTimeMarchingSolver(m)

adaptation_timestep = 1e-3
adaptation_final_time = 0.1
num_adaptation_steps = int(adaptation_final_time / adaptation_timestep)

In [ ]:
for step in tqdm.trange(num_adaptation_steps):
    monge_ampere_solver.step(adaptation_timestep)

In [ ]:
X = monge_ampere_solver.compute_coordinates()
new_mesh = firedrake.Mesh(X)

In [ ]:
fig, axes = icepack.plot.subplots()
icepack.plot.triplot(new_mesh, axes=axes)

### Once more, with feeling

In [ ]:
x = firedrake.SpatialCoordinate(new_mesh)
h_expr, u_expr = make_symbolic_input_data(x)

In [ ]:
Q1 = firedrake.FunctionSpace(new_mesh, family='CG', degree=1)
V1 = firedrake.VectorFunctionSpace(new_mesh, family='CG', degree=1)

Q2 = firedrake.FunctionSpace(new_mesh, family='CG', degree=2)
V2 = firedrake.VectorFunctionSpace(new_mesh, family='CG', degree=2)

fields1 = {
    'thickness': firedrake.interpolate(h_expr, Q1),
    'velocity': firedrake.interpolate(u_expr, V1)
}

fields2 = {
    'thickness': firedrake.interpolate(h_expr, Q2),
    'velocity': firedrake.interpolate(u_expr, V2)
}

In [ ]:
solver1 = icepack.solvers.FlowSolver(model, dirichlet_ids=[1])
solver2 = icepack.solvers.FlowSolver(model, dirichlet_ids=[1])

In [ ]:
h1 = fields1['thickness'].copy(deepcopy=True)
fields1['velocity'] = solver1.diagnostic_solve(**fields1, fluidity=A)

h2 = fields2['thickness'].copy(deepcopy=True)
fields2['velocity'] = solver2.diagnostic_solve(**fields2, fluidity=A)

In [ ]:
final_time = 400.
num_timesteps = 200
dt = final_time / num_timesteps
a = firedrake.Constant(0.0)

for step in tqdm.trange(num_timesteps):
    fields1['thickness'] = solver1.prognostic_solve(
        dt, **fields1, accumulation=a, thickness_inflow=h1
    )
    
    fields1['velocity'] = solver1.diagnostic_solve(
        **fields1, fluidity=A
    )
    
    fields2['thickness'] = solver2.prognostic_solve(
        dt, **fields2, accumulation=a, thickness_inflow=h2
    )
    
    fields2['velocity'] = solver2.diagnostic_solve(
        **fields2, fluidity=A
    )

In [ ]:
δh = firedrake.interpolate(fields2['thickness'] - fields1['thickness'], Q2)

In [ ]:
fig, axes = icepack.plot.subplots()
triangles = icepack.plot.tripcolor(
    δh, vmin=-10, vmax=+10, shading='gouraud', cmap='RdBu_r', axes=axes
)
fig.colorbar(triangles)

In [ ]:
volume = firedrake.assemble(fields2['thickness']**2 * dx)
firedrake.assemble(δh**2 * dx) / volume